# Mamba3Yolo — Kaggle Runbook

**First Mamba-3 (complex-state, exponential-trapezoidal) object detector + intrinsic controllability-Gramian explainability.**

Before running: **Settings → Accelerator: GPU (T4/P100)** and **Internet: ON**.

This notebook is self-contained. It clones the repo, then overwrites the core files with the
**verified** versions (the GitHub repo may lag behind). It then:
1. runs the parity state-tracking gate (proof the Mamba-3 core is real),
2. smoke-trains on `coco8` through the real Ultralytics pipeline,
3. trains **baseline vs ours** + mechanism ablations (RoPE / trapezoidal),
4. demos the Gramian explainability on the trained model,
5. collects results into tables.

> Honesty: report only numbers you measure. `official_mamba3_kernels=False` (no released kernel — the pure-PyTorch verified core runs). FLOPs are ~3× the baseline; state it.

## 0. Environment

**Run this install cell FIRST, before importing anything.** It floors `numpy>=2` so the
Ultralytics install can't leave a mixed numpy 1.x/2.x state (the `numpy.dtype size changed,
Expected 96 got 88` ABI error). If you still hit that error, do **Run → Restart & Run All**
once — Kaggle needs a kernel restart to load the reinstalled numpy.

In [ ]:
!pip install -q "numpy>=2.0" ultralytics==8.3.0 pymupdf

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
import os
if not os.path.exists('Mamba3Yolo'):
    !git clone -q https://github.com/ShMazumder/Mamba3Yolo.git
os.chdir('Mamba3Yolo')
os.makedirs('src/xai', exist_ok=True); os.makedirs('scripts', exist_ok=True)
print('cwd:', os.getcwd())

## 1. Patch in the verified source files
The repo on GitHub predates this work; these cells write the real, verified code.

In [ ]:
%%writefile src/blocks/mamba3_ref.py
"""
Faithful pure-PyTorch reference for the Mamba-3 selective SSM.

Implements the actual Mamba-3 recurrence (Lahoti et al., 2026, arXiv:2603.15569),
NOT a gated-MLP stand-in. Sequential O(L) scan: correct and honest, not fast.
Use it as the reference path when the official CUDA kernel is unavailable, and to
run the paper's ablations (trapezoidal on/off via lambda, RoPE on/off).

Recurrence (Eq. 5-6, 11):
    alpha_t = exp(dt_t * A)                         # decay, A < 0
    beta_t  = (1 - lambda_t) * dt_t * alpha_t       # previous-input (trapezoidal) weight
    gamma_t = lambda_t * dt_t                        # current-input weight
    h_t = alpha_t * h_{t-1}
          + beta_t  * outer(x_{t-1}, RoPE(B_{t-1}, Phi_{t-1}))
          + gamma_t * outer(x_t,     RoPE(B_t,     Phi_t))
    y_t = < h_t , RoPE(C_t, Phi_t) >                # sum over state dim
where Phi_t is the cumulative, dt-scaled rotary angle (the product of R_i in Eq 11,
realised via the RoPE trick: rotate B at write-time, C at read-time).

lambda_t = 1  ->  exponential-Euler  ==  Mamba-2 (single input term). This is the
knob for the trapezoidal ablation. SISO here; MIMO (rank-r B/C) is a documented
extension below, deliberately not enabled so we never over-claim what runs.
"""
from __future__ import annotations
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor


def _rope(v: Tensor, phi: Tensor) -> Tensor:
    """Rotate the state dimension of v in 2-D blocks by angles phi.

    v:   (..., N)      N even; consecutive pairs form the 2-D rotation blocks.
    phi: (..., N // 2) rotation angle per block.
    Returns v rotated, same shape as v. This is the standard RoPE trick, i.e. a
    real realisation of the complex-valued state transition (Prop. Complex-to-Real).
    """
    v2 = v.unflatten(-1, (-1, 2))          # (..., N/2, 2)
    cos, sin = torch.cos(phi), torch.sin(phi)
    x0, x1 = v2[..., 0], v2[..., 1]
    r0 = x0 * cos - x1 * sin
    r1 = x0 * sin + x1 * cos
    return torch.stack((r0, r1), dim=-1).flatten(-2)   # (..., N)


class Mamba3RefSSM(nn.Module):
    """Selective SSM with exponential-trapezoidal discretization + complex (RoPE) states.

    Diagonal (SISO) state per channel. Input/output are (B, L, D).
    """

    def __init__(
        self,
        d_model: int,
        d_state: int = 64,
        expand: int = 2,
        headdim: int = 64,
        dt_min: float = 1e-3,
        dt_max: float = 1e-1,
        rope_base: float = 10000.0,
        use_rope: bool = True,
        trapezoidal: bool = True,
    ):
        super().__init__()
        self.use_rope = use_rope        # False -> real-valued diagonal SSM (ablation control)
        self.trapezoidal = trapezoidal  # False -> lambda=1 = exponential-Euler = Mamba-2
        self.parallel = True            # chunked parallel scan; set False for O(L) reference
        assert d_state % 2 == 0, "d_state must be even for 2-D rotary blocks"
        self.d_model = d_model
        self.d_inner = int(expand * d_model)
        self.headdim = min(headdim, self.d_inner)
        assert self.d_inner % self.headdim == 0, "d_inner must be divisible by headdim"
        self.nheads = self.d_inner // self.headdim
        self.d_state = d_state

        # input / gate / output projections
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)   # x, z(gate)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        # data-dependent params: dt (per head), B/C (shared d_state), lambda (per head)
        self.dt_proj = nn.Linear(self.d_inner, self.nheads, bias=True)
        self.B_proj = nn.Linear(self.d_inner, d_state, bias=True)   # explicit B bias term
        self.C_proj = nn.Linear(self.d_inner, d_state, bias=True)   # explicit C bias term
        self.lam_proj = nn.Linear(self.d_inner, self.nheads, bias=True)
        # Rotation RATE (imaginary part of complex A) -- data-dependent, decoupled from
        # the small real decay step dt. Must reach ~pi/step to represent parity-style
        # rotational state tracking; that is why it is NOT tied to dt (see parity gate).
        self.w_proj = nn.Linear(self.d_inner, self.nheads, bias=True)

        # A < 0 via -exp(A_log); one decay rate per (head, state)
        self.A_log = nn.Parameter(torch.zeros(self.nheads, d_state))
        self.norm = nn.LayerNorm(self.d_inner)

        # fixed rotary base frequencies per 2-D block (data-independent part of the angle)
        theta = rope_base ** (-torch.arange(0, d_state, 2).float() / d_state)  # (N/2,)
        self.register_buffer("theta", theta, persistent=False)

        # init dt bias so softplus(dt) lands in [dt_min, dt_max]
        with torch.no_grad():
            dt = torch.empty(self.nheads).uniform_(math.log(dt_min), math.log(dt_max)).exp()
            self.dt_proj.bias.copy_(dt + torch.log(-torch.expm1(-dt)))  # inverse softplus

    def forward(self, x: Tensor) -> Tensor:
        B, L, _ = x.shape
        H, P, N = self.nheads, self.headdim, self.d_state

        xz = self.in_proj(x)
        u, z = xz.chunk(2, dim=-1)                     # (B, L, d_inner) each
        u = self.norm(u)

        dt = F.softplus(self.dt_proj(u))               # (B, L, H)  > 0
        lam = torch.sigmoid(self.lam_proj(u))          # (B, L, H)  in [0,1]
        if not self.trapezoidal:
            lam = torch.ones_like(lam)                 # Euler / Mamba-2 ablation
        Bt = self.B_proj(u)                            # (B, L, N)
        Ct = self.C_proj(u)                            # (B, L, N)
        A = -torch.exp(self.A_log)                     # (H, N)  < 0

        uh = u.view(B, L, H, P)                        # (B,L,H,P) per-head channels

        # cumulative rotary angle Phi_t = sum_{i<=t} w_i * theta   (vectorized over L)
        # w_t is the data-dependent rotation rate (imag part of A), free to reach ~pi.
        w = self.w_proj(u)                             # (B,L,H) unbounded rotation rate
        ang = w.unsqueeze(-1) * self.theta             # (B,L,H,N/2)
        phi = torch.cumsum(ang, dim=1)                 # (B,L,H,N/2)

        # trapezoidal discretization coeffs, all steps at once
        alpha = torch.exp(dt.unsqueeze(-1) * A)        # (B,L,H,N)
        beta = ((1 - lam) * dt).unsqueeze(-1) * alpha  # (B,L,H,N)
        gamma = (lam * dt).unsqueeze(-1)               # (B,L,H,1)

        Bt_h = Bt.unsqueeze(2).expand(B, L, H, N)
        Ct_h = Ct.unsqueeze(2).expand(B, L, H, N)
        if self.use_rope:
            Brot = _rope(Bt_h, phi)                    # (B,L,H,N)
            Crot = _rope(Ct_h, phi)
        else:
            Brot, Crot = Bt_h, Ct_h                    # real-valued control: no rotation

        # per-step input drive U_t = gamma_t (x_t (x) B_t) + beta_t (x_{t-1} (x) B_{t-1})
        cur = gamma.unsqueeze(3) * (uh.unsqueeze(-1) * Brot.unsqueeze(3))       # (B,L,H,P,N)
        prev_uh = F.pad(uh, (0, 0, 0, 0, 1, 0))[:, :L]      # shift right by 1 along L
        prev_Brot = F.pad(Brot, (0, 0, 0, 0, 1, 0))[:, :L]
        prv = beta.unsqueeze(3) * (prev_uh.unsqueeze(-1) * prev_Brot.unsqueeze(3))
        U = cur + prv                                  # (B,L,H,P,N)

        alpha5 = alpha.unsqueeze(3)                    # (B,L,H,1,N) decay, broadcasts over P
        Hs = self._scan_chunked(alpha5, U) if self.parallel else self._scan_reference(alpha5, U)

        y = (Hs * Crot.unsqueeze(3)).sum(-1)           # (B,L,H,P)  readout y_t = <Crot_t, h_t>
        y = y.reshape(B, L, self.d_inner)
        y = y * F.silu(z)                              # gate
        return self.out_proj(y)

    @staticmethod
    def _scan_reference(alpha: Tensor, U: Tensor) -> Tensor:
        """Sequential O(L) linear scan h_t = alpha_t h_{t-1} + U_t. Ground truth."""
        B, L, H, P, N = U.shape
        h = U.new_zeros(B, H, P, N)
        out = []
        for t in range(L):
            h = alpha[:, t] * h + U[:, t]
            out.append(h)
        return torch.stack(out, dim=1)                 # (B,L,H,P,N)

    @staticmethod
    def _scan_chunked(alpha: Tensor, U: Tensor, chunk: int = 32) -> Tensor:
        """Chunked prefix scan of h_t = alpha_t h_{t-1} + U_t.

        Within-chunk work is fully parallel via cumulative products; only the
        chunk-carry recurrence loops (L/chunk steps). Numerically safe while the
        per-chunk decay product stays well above underflow -- true here because
        alpha = exp(dt*A) with small dt keeps alpha near 1.
        """
        B, L, H, P, N = U.shape
        pad = (chunk - L % chunk) % chunk
        if pad:
            U = F.pad(U, (0, 0, 0, 0, 0, 0, 0, pad))
            alpha = F.pad(alpha, (0, 0, 0, 0, 0, 0, 0, pad), value=1.0)
        Lp = L + pad
        nc = Lp // chunk
        Uc = U.view(B, nc, chunk, H, P, N)
        Ac = alpha.view(B, nc, chunk, H, 1, N)
        Pin = torch.cumprod(Ac, dim=2)                 # inclusive within-chunk decay
        intra = Pin * torch.cumsum(Uc / Pin, dim=2)    # (B,nc,chunk,H,P,N)
        Ptot = Pin[:, :, -1]                           # (B,nc,H,1,N) full-chunk decay
        last = intra[:, :, -1]                         # (B,nc,H,P,N) chunk-final state
        carry = U.new_zeros(B, H, P, N)
        carries = []
        for c in range(nc):                            # nc = L/chunk steps, vectorized within
            carries.append(carry)
            carry = Ptot[:, c] * carry + last[:, c]
        carries = torch.stack(carries, dim=1)          # (B,nc,H,P,N) state entering each chunk
        Hs = intra + Pin * carries.unsqueeze(2)        # add carried state, broadcast over chunk
        return Hs.view(B, Lp, H, P, N)[:, :L]

    def _reverse_energy(self, alpha: Tensor) -> Tensor:
        """G_tau[n] = sum_{t>=tau} (prod_{k=tau+1..t} alpha_k[n])^2 via the stable
        reverse recurrence G_tau = 1 + alpha_{tau+1}^2 G_{tau+1} (G stays O(1), no
        division/underflow). (B,L,H,N). Eval-only (saliency), so an O(L) loop is fine."""
        a2 = alpha ** 2
        L = a2.shape[1]
        G = torch.empty_like(a2)
        g = torch.zeros_like(a2[:, 0])                      # G_{tau+1}, starts at 0 (G_L)
        for t in range(L - 1, -1, -1):
            g = torch.ones_like(g) if t == L - 1 else 1.0 + a2[:, t + 1] * g
            G[:, t] = g
        return G

    @torch.no_grad()
    def token_saliency(self, x: Tensor) -> Tensor:
        """Intrinsic controllability-energy saliency per input token. Returns (B, L).
        Closed-form from the SSM internals, one pass, no backprop."""
        B, L, _ = x.shape
        H, P, N = self.nheads, self.headdim, self.d_state
        u, _ = self.in_proj(x).chunk(2, dim=-1)
        u = self.norm(u)
        dt = F.softplus(self.dt_proj(u))
        lam = torch.sigmoid(self.lam_proj(u))
        Bt = self.B_proj(u)
        A = -torch.exp(self.A_log)
        uh = u.view(B, L, H, P)
        alpha = torch.exp(dt.unsqueeze(-1) * A)             # (B,L,H,N)
        gamma = lam * dt                                    # (B,L,H)
        Bt_h = Bt.unsqueeze(2).expand(B, L, H, N)
        if self.use_rope:
            w = self.w_proj(u)
            phi = torch.cumsum(w.unsqueeze(-1) * self.theta, dim=1)
            Brot = _rope(Bt_h, phi)
        else:
            Brot = Bt_h
        G = self._reverse_energy(alpha)                     # (B,L,H,N)
        energy = (Brot ** 2 * G).sum(-1)                    # (B,L,H)
        s = (gamma ** 2) * (uh ** 2).sum(-1) * energy       # (B,L,H)
        return s.mean(-1)                                   # (B,L)


# MIMO extension (not enabled): make B_proj / C_proj emit (N * r) and reshape to
# (N, r) matrices, replace the outer product `xt (x) Brot` with a matmul over the
# rank-r channel group, and read out y_t = Crot^T h. Enable only once validated
# against the paper's state-size vs. perplexity ablation -- until then, SISO is
# what runs, and that is what any paper text should claim.


In [ ]:
%%writefile src/blocks/mamba3_odss.py
"""
Mamba3ODSSBlock - NaN-safe + permanent fallback version
"""
from __future__ import annotations
from typing import Optional
import torch
import torch.nn as nn
from torch import Tensor

try:
    from mamba_ssm.modules.mamba3 import Mamba3
    HAS_MAMBA3 = True
except ImportError:
    HAS_MAMBA3 = False

# Faithful pure-PyTorch Mamba-3 selective SSM (real recurrence, complex/RoPE states).
# Replaces the old gated-MLP Mamba3Reference as the reference/fallback path. Verified
# on the parity state-tracking gate (scripts/validate_parity.py).
from src.blocks.mamba3_ref import Mamba3RefSSM

class DropPath(nn.Module):
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob
    def forward(self, x: Tensor) -> Tensor:
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = x.new_empty(shape).bernoulli_(keep)
        return x * mask / keep

class LSBlock(nn.Module):
    def __init__(self, dim: int, kernel_size: int = 3):
        super().__init__()
        self.dw = nn.Conv2d(dim, dim, kernel_size, padding=kernel_size // 2, groups=dim, bias=False)
        self.bn = nn.BatchNorm2d(dim)
        self.pw = nn.Conv2d(dim, dim, 1, bias=False)
        self.act = nn.GELU()
    def forward(self, x: Tensor) -> Tensor:
        residual = x
        x = self.dw(x); x = self.bn(x); x = self.act(x); x = self.pw(x)
        return x + residual

class RGBlock(nn.Module):
    def __init__(self, dim: int, expansion: float = 2.0):
        super().__init__()
        hidden = int(dim * expansion)
        self.fc1 = nn.Conv2d(dim, hidden, 1, bias=False)
        self.fc2 = nn.Conv2d(dim, hidden, 1, bias=False)
        self.dw = nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden, bias=False)
        self.fc_out = nn.Conv2d(hidden, dim, 1, bias=False)
        self.act = nn.SiLU()
        self.norm = nn.BatchNorm2d(dim)
    def forward(self, x: Tensor) -> Tensor:
        residual = x
        x1 = self.fc1(x); x2 = self.fc2(x)
        x = self.act(x1) * x2; x = self.dw(x); x = self.fc_out(x)
        return self.norm(x + residual)

class Mamba3Reference(nn.Module):
    def __init__(self, d_model: int, d_state: int = 64, expand: int = 2, headdim: int = 64, is_mimo: bool = True, mimo_rank: int = 4, **kwargs):
        super().__init__()
        self.d_model = d_model
        self.d_inner = int(expand * d_model)
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(self.d_inner)
        self.act = nn.SiLU()
        self.scale = nn.Parameter(torch.ones(1) * 0.1)
    def forward(self, x: Tensor) -> Tensor:
        residual = x
        xz = self.in_proj(x)
        x, z = xz.chunk(2, dim=-1)
        y = self.act(z) * self.norm(x)
        return residual + self.scale * self.out_proj(y)

class Mamba3SS2D(nn.Module):
    def __init__(self, dim: int, d_state: int = 64, expand: int = 2, headdim: int = 64, is_mimo: bool = True, mimo_rank: int = 4, drop_path: float = 0.0, use_rope: bool = True, trapezoidal: bool = True):
        super().__init__()
        self.dim = dim
        self.norm = nn.LayerNorm(dim)
        self.drop_path = DropPath(drop_path) if drop_path > 0 else nn.Identity()
        # Real Mamba-3 SSM reference (verified on parity gate), not the old gated MLP.
        # d_state forced even for the 2-D rotary blocks. use_rope/trapezoidal are the
        # mechanism-ablation toggles (complex-state and trapezoidal discretization).
        self.ref = Mamba3RefSSM(dim, d_state=d_state + (d_state % 2), expand=expand, headdim=headdim, use_rope=use_rope, trapezoidal=trapezoidal)
        self._use_official = False
        self.mamba = None
        if HAS_MAMBA3:
            try:
                self.mamba = Mamba3(d_model=dim, d_state=min(d_state, 64), expand=expand, headdim=min(headdim, 64), is_mimo=False, mimo_rank=1)
                self._use_official = True
            except Exception:
                self.mamba = self.ref
                self._use_official = False
        else:
            self.mamba = self.ref
            self._use_official = False
    def _safe_mamba(self, seq: Tensor) -> Tensor:
        seq = seq.contiguous()
        if not self._use_official:
            return self.ref(seq)
        try:
            return self.mamba(seq)
        except Exception:
            if self._use_official:
                print("[Mamba3SS2D] Official kernel failed once → permanently using pure-PyTorch reference")
                self._use_official = False
                self.mamba = None
            return self.ref(seq)
    def _four_dir_scan(self, x: Tensor) -> Tensor:
        B, C, H, W = x.shape
        x = x.contiguous()
        seq_h = x.flatten(2).transpose(1, 2).contiguous()
        seq_hf = torch.flip(seq_h, dims=[1]).contiguous()
        seq_v = x.permute(0, 1, 3, 2).contiguous().flatten(2).transpose(1, 2).contiguous()
        seq_vf = torch.flip(seq_v, dims=[1]).contiguous()
        out_h = self._safe_mamba(seq_h)
        out_hf = self._safe_mamba(seq_hf)
        out_v = self._safe_mamba(seq_v)
        out_vf = self._safe_mamba(seq_vf)
        out_h = out_h.transpose(1, 2).contiguous().view(B, C, H, W)
        out_hf = torch.flip(out_hf, dims=[1]).transpose(1, 2).contiguous().view(B, C, H, W)
        out_v = out_v.transpose(1, 2).contiguous().view(B, C, W, H).permute(0, 1, 3, 2).contiguous()
        out_vf = torch.flip(out_vf, dims=[1]).transpose(1, 2).contiguous().view(B, C, W, H).permute(0, 1, 3, 2).contiguous()
        return (out_h + out_hf + out_v + out_vf) * 0.25
    def forward(self, x: Tensor) -> Tensor:
        residual = x
        x = x.permute(0, 2, 3, 1).contiguous()
        x = self.norm(x)
        x = x.permute(0, 3, 1, 2).contiguous()
        x = self._four_dir_scan(x)
        return residual + self.drop_path(x)

    @torch.no_grad()
    def spatial_saliency(self, x: Tensor) -> Tensor:
        """Intrinsic controllability-Gramian saliency map for this block. (B,H,W).
        Runs the 4 scan directions and folds each token-saliency back to 2D."""
        B, C, H, W = x.shape
        x = x.permute(0, 2, 3, 1).contiguous()
        x = self.norm(x)
        x = x.permute(0, 3, 1, 2).contiguous()
        seq_h = x.flatten(2).transpose(1, 2).contiguous()
        seq_hf = torch.flip(seq_h, dims=[1]).contiguous()
        seq_v = x.permute(0, 1, 3, 2).contiguous().flatten(2).transpose(1, 2).contiguous()
        seq_vf = torch.flip(seq_v, dims=[1]).contiguous()
        sal = self.ref.token_saliency if hasattr(self.ref, "token_saliency") else None
        if sal is None:      # official-kernel path has no intrinsic saliency
            return x.new_zeros(B, H, W)
        mh = sal(seq_h).view(B, H, W)
        mhf = torch.flip(sal(seq_hf), dims=[1]).view(B, H, W)
        mv = sal(seq_v).view(B, W, H).permute(0, 2, 1)
        mvf = torch.flip(sal(seq_vf), dims=[1]).view(B, W, H).permute(0, 2, 1)
        return (mh + mhf + mv + mvf) * 0.25

class Mamba3ODSSBlock(nn.Module):
    def __init__(self, dim: int, d_state: int = 64, expand: int = 2, headdim: int = 64, is_mimo: bool = True, mimo_rank: int = 4, drop_path: float = 0.0, mlp_ratio: float = 2.0, use_rope: bool = True, trapezoidal: bool = True):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(dim, dim, 1, bias=False), nn.BatchNorm2d(dim), nn.SiLU(inplace=True))
        self.ls = LSBlock(dim)
        self.ss2d = Mamba3SS2D(dim=dim, d_state=d_state, expand=expand, headdim=headdim, is_mimo=is_mimo, mimo_rank=mimo_rank, drop_path=drop_path, use_rope=use_rope, trapezoidal=trapezoidal)
        self.rg = RGBlock(dim, expansion=mlp_ratio)
    def forward(self, x: Tensor) -> Tensor:
        x = self.conv1(x); x = self.ls(x); x = self.ss2d(x); x = self.rg(x)
        return x

def build_mamba3_odss(c1: int, c2: int, n: int = 1, **kwargs) -> nn.Module:
    if c1 != c2:
        return nn.Sequential(nn.Conv2d(c1, c2, 1, bias=False), nn.BatchNorm2d(c2), nn.SiLU(inplace=True), *[Mamba3ODSSBlock(c2, **kwargs) for _ in range(n)])
    return nn.Sequential(*[Mamba3ODSSBlock(c2, **kwargs) for _ in range(n)])


class Mamba3ODSS(nn.Module):
    """Ultralytics-compatible wrapper (c1, c2, n, ...) so a YOLO yaml can reference
    `Mamba3ODSS` where it would use C3k2. Matches the channel+repeat calling
    convention Ultralytics' parse_model uses for CSP blocks. Leaner defaults
    (expand=1, mlp_ratio=1.0, d_state=32) keep the -s variant near ~12-13M."""

    def __init__(self, c1: int, c2: int, n: int = 1, d_state: int = 32,
                 expand: int = 1, mlp_ratio: float = 1.0,
                 use_rope: bool = True, trapezoidal: bool = True):
        super().__init__()
        self.block = build_mamba3_odss(c1, c2, n=max(int(n), 1), d_state=d_state,
                                       expand=expand, mlp_ratio=mlp_ratio,
                                       use_rope=bool(use_rope), trapezoidal=bool(trapezoidal))

    def forward(self, x: Tensor) -> Tensor:
        return self.block(x)


In [ ]:
%%writefile scripts/ultra_mamba3.py
#!/usr/bin/env python3
"""
Register Mamba3ODSS with Ultralytics so a YOLO yaml can use it, and generate a
yolo11-mamba3 config (C3k2 mixers -> Mamba3ODSS). This makes the Mamba-3 block
survive Ultralytics' yaml rebuild during .train() -- the surgery-on-object route
does NOT (train() re-parses from cfg).

    from scripts.ultra_mamba3 import register, make_yaml
    register()
    from ultralytics import YOLO
    model = YOLO(make_yaml("s"))          # yolo11s with Mamba-3 mixers
    model.train(data="coco.yaml", ...)    # real DFL loss + assigner + NMS + mAP

No site-packages files are modified: parse_model is re-bound in-process.
"""
from __future__ import annotations
import sys, inspect
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))


def register(verbose: bool = True) -> None:
    """Patch ultralytics.nn.tasks.parse_model so it treats Mamba3ODSS as a
    channel-changing, repeatable CSP block (like C3k2)."""
    import ultralytics.nn.tasks as T
    from src.blocks.mamba3_odss import Mamba3ODSS
    if getattr(T, "_mamba3_registered", False):
        return

    src = inspect.getsource(T.parse_model)
    a_rep = "args.insert(2, n)"           # the repeat-insert set sits just before this
    a_chan = "c1, c2 = ch[f], args[0]"    # the channel set sits just before this
    if a_rep not in src or a_chan not in src:
        raise RuntimeError("parse_model anchors not found; ultralytics layout changed "
                           "(pinned 8.3.0). Update scripts/ultra_mamba3.py.")

    def inject(text: str, anchor: str) -> str:
        i = text.index(anchor)
        brace = text.rindex("}", 0, i)    # closing brace of the set literal before anchor
        return text[:brace] + "    Mamba3ODSS,\n            " + text[brace:]

    src = inject(src, a_rep)
    src = inject(src, a_chan)

    T.Mamba3ODSS = Mamba3ODSS                       # visible to the set literal at call time
    exec(compile(src, T.__file__, "exec"), T.__dict__)   # re-bind parse_model in-place
    T._mamba3_registered = True
    if verbose:
        print("registered Mamba3ODSS with ultralytics parse_model")


def make_yaml(scale: str = "s", d_state: int = 32, expand: int = 1,
              mlp_ratio: float = 1.0, use_rope: bool = True, trapezoidal: bool = True,
              tag: str | None = None, out: str | None = None) -> str:
    """Generate a yolo11-mamba3 yaml from the stock yolo11 config, swapping every
    C3k2 mixer for Mamba3ODSS. Ablation flags are baked into each block's args:
    [c2, d_state, expand, mlp_ratio, use_rope, trapezoidal]. Returns the path."""
    import ultralytics, yaml
    stock = Path(ultralytics.__file__).parent / "cfg/models/11/yolo11.yaml"
    d = yaml.safe_load(stock.read_text())
    extra = [int(d_state), int(expand), float(mlp_ratio), bool(use_rope), bool(trapezoidal)]

    def swap(layers):
        for layer in layers:                       # layer = [from, number, module, args]
            if layer[2] == "C3k2":
                layer[2] = "Mamba3ODSS"
                layer[3] = [layer[3][0], *extra]   # out-channels + ablation args
        return layers

    d["backbone"] = swap(d["backbone"])
    d["head"] = swap(d["head"])

    if tag is None:
        tag = "" if (use_rope and trapezoidal) else \
              ("-norope" if not use_rope else "") + ("-euler" if not trapezoidal else "")
    out_dir = Path(__file__).resolve().parents[1] / "configs/models"
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out or str(out_dir / f"yolo11{scale}-mamba3{tag}.yaml")
    Path(out).write_text(yaml.safe_dump(d, sort_keys=False))
    return out


if __name__ == "__main__":
    register()
    path = make_yaml("s")
    print("wrote", path)
    from ultralytics import YOLO
    import torch
    from src.blocks.mamba3_ref import Mamba3RefSSM
    m = YOLO(path)
    n = sum(isinstance(mm, Mamba3RefSSM) for mm in m.model.modules())
    p = sum(pp.numel() for pp in m.model.parameters()) / 1e6
    print(f"YOLO({Path(path).name}): Mamba3RefSSM blocks={n}, params={p:.2f}M")
    with torch.no_grad():
        out = m.model(torch.randn(1, 3, 320, 320))
    print("forward ok" if out is not None else "forward FAILED")


In [ ]:
%%writefile src/xai/gramian.py
"""
Intrinsic controllability-Gramian explainability for Mamba3Yolo.

Unlike Grad-CAM (gradient-based, saturates on recurrent nets) this reads the
attribution directly from the SSM's own dynamics -- one forward pass, no backprop,
O(L). It is closed-form precisely because Mamba-3's state is complex-diagonal.

  from src.xai.gramian import gramian_saliency
  heat = gramian_saliency(model, img)     # (B, H, W) in [0,1], per input pixel

This is the paper's novel XAI contribution. Compare against Grad-CAM++ (src/xai/gradcam.py)
on insertion/deletion + pointing-game -- the numbers competitors (e.g. AKCMamba-YOLO,
which only shows qualitative Grad-CAM) never report.
"""
from __future__ import annotations
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from typing import Optional
import torch
import torch.nn.functional as F
from torch import Tensor

from src.blocks.mamba3_odss import Mamba3SS2D


def _norm(t: Tensor) -> Tensor:
    B = t.shape[0]
    flat = t.view(B, -1)
    lo = flat.min(1, keepdim=True).values
    hi = flat.max(1, keepdim=True).values
    return ((flat - lo) / (hi - lo + 1e-8)).view_as(t)


@torch.no_grad()
def gramian_saliency(model: torch.nn.Module, x: Tensor,
                     size: Optional[tuple] = None) -> Tensor:
    """Aggregate intrinsic saliency over every Mamba3SS2D block. Returns (B, H, W)."""
    blocks = [m for m in model.modules() if isinstance(m, Mamba3SS2D)]
    if not blocks:
        raise ValueError("no Mamba3SS2D blocks in model")
    captured: dict = {}
    hooks = [b.register_forward_pre_hook(
        lambda mod, inp, key=b: captured.__setitem__(key, inp[0].detach())) for b in blocks]
    was_training = model.training
    model.eval()
    model(x)
    for h in hooks:
        h.remove()
    if was_training:
        model.train()

    H0, W0 = size or x.shape[-2:]
    total = None
    for b, inp in captured.items():
        sal = b.spatial_saliency(inp)                      # (B,h,w)
        sal = F.interpolate(sal.unsqueeze(1), size=(H0, W0),
                            mode="bilinear", align_corners=False).squeeze(1)
        sal = _norm(sal)
        total = sal if total is None else total + sal
    return _norm(total)


# ---------------------------------------------------------------- self-tests
if __name__ == "__main__":
    import sys
    from pathlib import Path
    sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
    from src.blocks.mamba3_ref import Mamba3RefSSM

    torch.manual_seed(0)

    # 1) reverse-energy G matches brute force
    ssm = Mamba3RefSSM(d_model=24, d_state=8, headdim=12).eval()
    B, L, H, N = 2, 40, ssm.nheads, ssm.d_state
    alpha = torch.rand(B, L, H, N) * 0.3 + 0.6            # decays in [0.6,0.9]
    G_fast = ssm._reverse_energy(alpha)
    G_brute = torch.zeros_like(alpha)
    for tau in range(L):                                  # G_tau = sum_{t>=tau} (prod alpha)^2
        D = torch.ones(B, H, N)
        for t in range(tau, L):
            if t > tau:
                D = D * alpha[:, t]
            G_brute[:, tau] += D ** 2
    err = (G_fast - G_brute).abs().max().item()
    print(f"[reverse-energy] max|fast-brute| = {err:.2e}  ->", "OK" if err < 1e-3 else "MISMATCH")

    # 2) does the block's saliency localize a bright region?
    ss2d = Mamba3SS2D(dim=16, d_state=8, headdim=16).eval()
    feat = torch.randn(1, 16, 16, 16) * 0.1
    feat[:, :, 3:7, 10:14] += 3.0                         # bright patch top-right
    sal = ss2d.spatial_saliency(feat)[0]                  # (16,16)
    patch = sal[3:7, 10:14].mean().item()
    elsewhere = (sal.sum() - sal[3:7, 10:14].sum()) / (sal.numel() - 16)
    print(f"[block saliency] patch={patch:.3f} vs elsewhere={elsewhere.item():.3f}  ->",
          "LOCALIZES" if patch > 2 * elsewhere.item() else "weak")

    # 3) end-to-end on the integrated YOLO11-mamba3
    try:
        from scripts.ultra_mamba3 import register, make_yaml
        from ultralytics import YOLO
        register(verbose=False)
        model = YOLO(make_yaml("s")).model
        img = torch.randn(1, 3, 128, 128) * 0.1
        img[:, :, 20:44, 80:104] += 2.0                  # bright object
        heat = gramian_saliency(model, img)              # (1,128,128)
        obj = heat[0, 20:44, 80:104].mean().item()
        bg = (heat[0].sum() - heat[0, 20:44, 80:104].sum()) / (heat[0].numel() - 24 * 24)
        print(f"[full model] heat shape {tuple(heat.shape)}, obj={obj:.3f} vs bg={bg.item():.3f}  ->",
              "OBJECT-FOCUSED" if obj > bg.item() else "diffuse")
    except Exception as e:
        print("[full model] skipped:", type(e).__name__, e)


In [ ]:
%%writefile scripts/validate_parity.py
"""
Honesty gate for Mamba3RefSSM: the parity / state-tracking test.

Mamba-3 (arXiv:2603.15569, Sec 3.2) claims complex-valued (rotational) states
solve running parity  y_t = (sum_{i<=t} x_i) mod 2  -- a task real-valued diagonal
SSMs provably CANNOT (Grazzi et al., Thm 1: real eigenvalues can't do rotation).

Decisive result is the CONTRAST, not the absolute:
  - use_rope=True  (complex/rotational)  -> should learn parity  (acc -> ~1.0)
  - use_rope=False (real diagonal SSM)   -> should FAIL           (acc ~ 0.5)
If both pass, the rotation isn't doing the work (wiring bug). If the True model
fails, there is no Mamba-3 state-tracking claim. Either way we learn the truth.

CPU-friendly: small model, short sequences, a few hundred steps.
"""
import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import torch
import torch.nn as nn
from src.blocks.mamba3_ref import Mamba3RefSSM


class ParityNet(nn.Module):
    def __init__(self, use_rope: bool, d_model=32, d_state=16, headdim=16):
        super().__init__()
        self.embed = nn.Linear(1, d_model)
        self.ssm = Mamba3RefSSM(d_model, d_state=d_state, headdim=headdim, use_rope=use_rope)
        self.head = nn.Linear(d_model, 1)

    def forward(self, bits):                 # bits: (B, L) in {0,1}
        h = self.embed(bits.unsqueeze(-1))   # (B, L, d_model)
        h = self.ssm(h)
        return self.head(h).squeeze(-1)      # (B, L) per-position logit


def batch(B, L, device):
    bits = torch.randint(0, 2, (B, L), device=device).float()
    target = torch.cumsum(bits, dim=1) % 2   # running parity, per position
    return bits, target


def run(use_rope, steps=800, B=128, L=24, seed=0, device="cpu"):
    torch.manual_seed(seed)
    net = ParityNet(use_rope).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    lossf = nn.BCEWithLogitsLoss()
    for s in range(steps):
        bits, tgt = batch(B, L, device)
        opt.zero_grad()
        loss = lossf(net(bits), tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
    # eval on fresh data, same length + a longer length (length generalization)
    net.eval()
    with torch.no_grad():
        accs = {}
        for evalL in (L, 2 * L):
            bits, tgt = batch(512, evalL, device)
            pred = (net(bits) > 0).float()
            accs[evalL] = (pred == tgt).float().mean().item()
    return accs


if __name__ == "__main__":
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device={dev}\nrunning parity (cumsum mod 2), per-position accuracy\n")
    on = run(use_rope=True, device=dev)
    off = run(use_rope=False, device=dev)
    L = 24
    print(f"  RoPE ON  (complex): L={L} acc={on[L]:.3f}   L={2*L} acc={on[2*L]:.3f}")
    print(f"  RoPE OFF (real)   : L={L} acc={off[L]:.3f}   L={2*L} acc={off[2*L]:.3f}")
    print()
    gap = on[L] - off[L]
    if on[L] > 0.9 and off[L] < 0.75:
        print(f"  PASS: complex solves parity, real fails (gap {gap:+.3f}). Mamba-3 state-tracking is REAL.")
    elif on[L] > 0.9 and off[L] > 0.9:
        print(f"  SUSPECT: both solve it (gap {gap:+.3f}). Rotation may not be load-bearing -- inspect wiring.")
    elif on[L] < 0.75:
        print(f"  FAIL: complex model cannot track parity (acc {on[L]:.3f}). Claim #2 (Mamba-3 core) is NOT supported.")
    else:
        print(f"  WEAK: gap {gap:+.3f}. Inconclusive -- tune steps/lr and rerun before trusting.")


In [ ]:
import sys; sys.path.insert(0, os.getcwd())
print('patched files in place')

## 2. Honesty gate — parity / state-tracking
Complex (RoPE) state must solve running-parity (~1.0); the real-valued control must fail (~0.5). This is the proof the Mamba-3 core actually tracks state.

In [ ]:
!python scripts/validate_parity.py

## 3. Smoke test — real Ultralytics pipeline on coco8
Confirms the Mamba-3 model trains through the genuine DFL loss + assigner + NMS + COCO mAP val, and **survives** Ultralytics' yaml rebuild (surgery-on-object does not).

In [ ]:
from scripts.ultra_mamba3 import register, make_yaml
from ultralytics import YOLO
from src.blocks.mamba3_ref import Mamba3RefSSM
register()
y = YOLO(make_yaml('s'))
print('Mamba blocks before train:', sum(isinstance(m, Mamba3RefSSM) for m in y.model.modules()))
y.train(data='coco8.yaml', epochs=3, imgsz=320, batch=8, device=0, workers=2,
        plots=False, verbose=False, exist_ok=True, name='coco8_smoke')
print('Mamba blocks after train :', sum(isinstance(m, Mamba3RefSSM) for m in y.model.modules()))

## 4. Main comparison + ablations (configure below)

Same base (YOLO11-s), same data/schedule — **only the mixer differs**.

**Kaggle note:** this model is ~69 GFLOPs (the O(L) scan is heavy). Full COCO/300 epochs will not
fit a single Kaggle session. Defaults below use `coco128` for a *real but small* demo that produces
genuine numbers. For paper numbers, point `DATA` at your full dataset and raise `EPOCHS` on adequate
compute (or resume across sessions).

In [ ]:
DATA   = 'coco128.yaml'   # <-- your data.yaml for real numbers (COCO / medical)
EPOCHS = 60
IMGSZ  = 512              # 640 for paper; 512 is faster on Kaggle T4
BATCH  = 16
DEV    = 0

### 4a. Baseline — stock YOLO11-s (C3k2 mixer)

In [ ]:
from ultralytics import YOLO
YOLO('yolo11s.yaml').train(data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEV,
                           cos_lr=True, plots=False, exist_ok=True, name='yolo11s_base')

### 4b. Ours + ablations — Mamba-3 full / no-RoPE / Euler(=Mamba-2)

In [ ]:
from scripts.ultra_mamba3 import register, make_yaml
from ultralytics import YOLO
register()
variants = {
    'mamba3_full':   make_yaml('s'),                    # RoPE + trapezoidal
    'mamba3_norope': make_yaml('s', use_rope=False),    # - complex state
    'mamba3_euler':  make_yaml('s', trapezoidal=False), # - trapezoidal (= Mamba-2)
}
for name, ycfg in variants.items():
    print('=== training', name, '===')
    YOLO(ycfg).train(data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEV,
                     cos_lr=True, plots=False, exist_ok=True, name=name)

## 5. Results table
Pulls final val metrics from each run's `results.csv`.

In [ ]:
import pandas as pd, glob, os
rows = []
for run in ['yolo11s_base', 'mamba3_full', 'mamba3_norope', 'mamba3_euler']:
    csv = f'runs/detect/{run}/results.csv'
    if os.path.exists(csv):
        df = pd.read_csv(csv); df.columns = [c.strip() for c in df.columns]
        last = df.iloc[-1]
        rows.append({'run': run,
                     'mAP50': round(float(last.get('metrics/mAP50(B)', float('nan'))), 4),
                     'mAP50-95': round(float(last.get('metrics/mAP50-95(B)', float('nan'))), 4)})
print(pd.DataFrame(rows).to_string(index=False) if rows else 'no runs yet')

## 6. Gramian explainability on the trained model
Intrinsic controllability-energy saliency — one forward pass, no backprop. Only meaningful on a **trained** model.

In [ ]:
import torch, numpy as np, glob
import matplotlib.pyplot as plt
from scripts.ultra_mamba3 import register
from ultralytics import YOLO
from src.xai.gramian import gramian_saliency
register()
ckpt = 'runs/detect/mamba3_full/weights/best.pt'
model = YOLO(ckpt).model.eval()

# grab one val image
imgs = sorted(glob.glob('datasets/coco128/images/train2017/*.jpg'))[:1] or \
       sorted(glob.glob('/kaggle/working/Mamba3Yolo/datasets/**/*.jpg', recursive=True))[:1]
from ultralytics.data.augment import LetterBox
import cv2
im0 = cv2.cvtColor(cv2.imread(imgs[0]), cv2.COLOR_BGR2RGB)
im = LetterBox((512, 512))(image=im0)
x = torch.from_numpy(im).permute(2, 0, 1).float()[None] / 255.0
with torch.no_grad():
    heat = gramian_saliency(model, x)[0].cpu().numpy()

fig, ax = plt.subplots(1, 3, figsize=(14, 5))
ax[0].imshow(im); ax[0].set_title('input'); ax[0].axis('off')
ax[1].imshow(heat, cmap='jet'); ax[1].set_title('Gramian saliency'); ax[1].axis('off')
ax[2].imshow(im); ax[2].imshow(heat, cmap='jet', alpha=0.5); ax[2].set_title('overlay'); ax[2].axis('off')
plt.tight_layout(); plt.show()

## 7. Integrity checklist
- Every reported number is measured here — no placeholders.
- `official_mamba3_kernels = False` (verified pure-PyTorch core).
- FLOPs ~3× baseline — report openly; scope any efficiency claim to what you measure.
- Ablations use the same schedule as the main model.
- Do **not** cite the old fabricated tables (Gemini report / `paper_summary.json`).

For the XAI faithfulness metrics (insertion/deletion, pointing-game) vs Grad-CAM++, and full-COCO
training, see `docs/EXPERIMENTS_RUNBOOK.md`.